In [7]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error, r2_score

# Load the dataset
df = pd.read_excel(r'../updated_fc_predictions.xlsx', sheet_name='Sheet1')
df.dropna(inplace=True)

# Define features and target
X = df[['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%']].values
y = df['fc (MPa)'].values
w_c = df['w/b'].values  # assuming 'w/b' is the water-to-cement ratio

# Define the full equation for fc = A * B^(-w/c)
def fc_model(X, *params):
    # Unpack parameters for A and B (linear terms)
    a_params = params[:len(X[0])]  # Parameters for A
    b_params = params[len(X[0]):]  # Parameters for B

    # Extract features
    AGE, PC, PC_TYPE, FA, SS, SF, FAGG, CAGG, WATER, AEA, WR_HR, WR, ACC, VOID, w_b, b_a, CAGG_perc, FAGG_perc, FA_perc, SS_perc, SF_perc = X.T

    # Compute A and B as linear combinations of the features
    A = np.dot(X, a_params)
    B = np.dot(X, b_params)

    # Return the model equation: fc = A * B^(-w/c)
    return A * B**(-w_b)

# Prepare the initial guess for the parameters
initial_guess = np.ones(2 * len(X[0]))  # One set of parameters for A and one set for B

# Fit the model using nonlinear least squares
popt, pcov = curve_fit(fc_model, X, y, p0=initial_guess)

# Extract the parameters for A and B
a_params = popt[:len(X[0])]
b_params = popt[len(X[0]):]

print("Parameters for A:", a_params)
print("Parameters for B:", b_params)

# Make predictions using the fitted parameters
y_pred = fc_model(X, *popt)

# Evaluate the model
mse = mean_squared_error(y, y_pred)
r2 = r2_score(y, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R² Score: {r2}")

columns = ['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%']
# Optional: Show the fitted equations for A and B
print("Equation for A: A = {:.4f}".format(popt[0]) + " + " + " + ".join([f"{coef:.4f}*{name}" for coef, name in zip(a_params, columns)]))
print("Equation for B: B = {:.4f}".format(popt[len(X[0])]) + " + " + " + ".join([f"{coef:.4f}*{name}" for coef, name in zip(b_params, columns)]))


/var/folders/m1/ws2562m105bdpyh7s345x7cw0000gn/T/ipykernel_22498/2237581241.py:30: RuntimeWarning: invalid value encountered in power
  return A * B**(-w_b)


Parameters for A: [ 2.02649979e+00  4.63007743e-01  1.85261401e+00 -8.78267861e-01
 -1.13638654e-01  5.79865066e-01  1.76086445e-02  1.36687695e-01
 -5.81030924e-01  2.22074305e+00  2.76762775e-01  2.35112838e-01
 -7.69213627e-03 -1.32311467e+00  8.92458565e+02  4.79381370e+02
 -8.84490263e+02 -6.28109398e+02  1.00933951e+03  5.14485965e+02
  3.61960023e+02]
Parameters for B: [ 3.71043510e-01 -5.42509113e-02  1.50506327e-01 -1.00791042e+00
 -3.77794461e-01  5.80179827e-01  1.08481604e-01  1.65837196e-01
 -6.11737624e-01  4.77110003e+00  1.56318100e-01  6.11122775e-02
 -1.55709814e-02 -5.20164121e-01  4.23265300e+02  1.32435230e+03
 -7.39641076e+02 -5.87959386e+02  6.94400831e+02  2.63348088e+02
 -2.82852949e+02]
Mean Squared Error: 93.43757236235767
R² Score: 0.5177373288000228
Equation for A: A = 2.0265 + 2.0265*AGE + 0.4630*PC + 1.8526*PC_TYPE + -0.8783*FA + -0.1136*SS + 0.5799*SF + 0.0176*FAGG + 0.1367*CAGG + -0.5810*WATER + 2.2207*AEA + 0.2768*WR_HR + 0.2351*WR + -0.0077*ACC + -1.3

In [9]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error, r2_score

# Load the dataset
df = pd.read_excel(r'../updated_fc_predictions.xlsx', sheet_name='Sheet1')
df.dropna(inplace=True)

# Define features and target
X = df[['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%']].values
y = df['fc (MPa)'].values
w_c = df['w/b'].values  # assuming 'w/b' is the water-to-cement ratio

# Add a column of ones to X to include a constant term (intercept)
X = np.hstack([np.ones((X.shape[0], 1)), X])

# Define the full equation for fc = A * B^(-w/c)
def fc_model(X, *params):
    # Unpack parameters for A and B (linear terms)
    a_params = params[:len(X[0])]  # Parameters for A
    b_params = params[len(X[0]):]  # Parameters for B

    # Compute A and B as linear combinations of the features
    A = np.dot(X, a_params)
    B = np.dot(X, b_params)

    # Avoid invalid values in B by ensuring B is strictly positive
    B = np.clip(B, 1e-6, None)  # Ensure B is positive

    # Return the model equation: fc = A * B^(-w/c)
    return A * B**(-X[:, 16])  # w/b is at index 16 after adding a column for intercept

# Prepare the initial guess for the parameters
initial_guess = np.ones(2 * len(X[0]))  # One set of parameters for A and one set for B

# Fit the model using nonlinear least squares and increase maxfev
popt, pcov = curve_fit(fc_model, X, y, p0=initial_guess, maxfev=20000)

# Extract the parameters for A and B
a_params = popt[:len(X[0])]
b_params = popt[len(X[0]):]

# Make predictions using the fitted parameters
y_pred = fc_model(X, *popt)

# Evaluate the model
mse = mean_squared_error(y, y_pred)
r2 = r2_score(y, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R² Score: {r2}")

# Optional: Show the fitted equations for A and B
columns = ['Intercept', 'AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
           'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%']

print("Equation for A: A = {:.4f}".format(popt[0]) + " + " + " + ".join([f"{coef:.4f}*{name}" for coef, name in zip(a_params, columns)]))
print("Equation for B: B = {:.4f}".format(popt[len(X[0])]) + " + " + " + ".join([f"{coef:.4f}*{name}" for coef, name in zip(b_params, columns)]))


Mean Squared Error: 94.50863816956297
R² Score: 0.5122091986896788
Equation for A: A = 1.9140 + 1.9140*Intercept + 0.1904*AGE + -0.0060*PC + 0.5414*PC_TYPE + -0.0493*FA + -0.0175*SS + 0.2814*SF + -0.0047*FAGG + 0.0134*CAGG + 0.0186*WATER + -0.2940*AEA + -0.0203*WR_HR + 0.0282*WR + 0.0191*ACC + -0.2628*VOID + -30.9441*w/b + 32.8238*b/a + -13.5333*CAGG% + 32.8574*FAGG% + 20.4856*FA% + 7.5624*SS% + -208.3472*SF%
Equation for B: B = -56.3841 + -56.3841*Intercept + -0.0000*AGE + -0.0008*PC + 0.0048*PC_TYPE + -0.0009*FA + -0.0005*SS + 0.0054*SF + -0.0002*FAGG + 0.0003*CAGG + 0.0001*WATER + -0.0015*AEA + -0.0004*WR_HR + 0.0001*WR + 0.0004*ACC + -0.0023*VOID + 0.2667*w/b + 1.7186*b/a + 55.7700*CAGG% + 57.2469*FAGG% + -0.1190*FA% + -0.3616*SS% + -5.0090*SF%


In [10]:
print("Parameters for A:", a_params)
print("Parameters for B:", b_params)

Parameters for A: [ 1.91400249e+00  1.90416036e-01 -6.04625137e-03  5.41377126e-01
 -4.92502110e-02 -1.75108619e-02  2.81439962e-01 -4.67507566e-03
  1.34341141e-02  1.86213462e-02 -2.94031336e-01 -2.02643193e-02
  2.81646631e-02  1.91139061e-02 -2.62797602e-01 -3.09440551e+01
  3.28237959e+01 -1.35333372e+01  3.28574407e+01  2.04856306e+01
  7.56244878e+00 -2.08347153e+02]
Parameters for B: [-5.63841149e+01 -8.88907152e-07 -7.64206150e-04  4.76004207e-03
 -9.48165598e-04 -4.51401528e-04  5.41376545e-03 -2.27336283e-04
  2.63054186e-04  8.89540589e-05 -1.46954995e-03 -3.97470701e-04
  1.43721900e-04  3.59391375e-04 -2.31797788e-03  2.66703984e-01
  1.71857668e+00  5.57699787e+01  5.72468554e+01 -1.18966024e-01
 -3.61569032e-01 -5.00899487e+00]
